## Install `holidays`

Databricks Serverless does not ship the `holidays` package. This cell installs it and
restarts Python so `src.features` can import it. Takes about five seconds.


In [ ]:
%pip install holidays -q
dbutils.library.restartPython()


# 03 — Silver: cleaned + temporally enriched

- Drops target-leakage and post-facto columns (actual times, taxi, delay causes).
- Drops 2020 (COVID anomaly).
- Preserves `arrival_delay` nulls (cancelled / diverted); Gold filters them.
- Adds calendar + US-federal-holiday features via a broadcast **date dimension**
  built from the unit-tested helpers in `src.features`.
- Enforces the row contract with Delta `CHECK` constraints, then compacts with
  `OPTIMIZE ... ZORDER`.

Idempotent: full overwrite.


In [ ]:
import sys
import time
from datetime import date

sys.path.append("..")

from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, dayofmonth, dayofweek, floor, quarter as spark_quarter,
    to_date, udf, weekofyear, when, year, month as spark_month,
)
from pyspark.sql.types import (
    DateType, IntegerType, StringType, StructField, StructType,
)

from src import config
from src.features import (
    build_date_dimension, check_holiday, check_holiday_period, check_near_holiday,
    get_season,
)


In [ ]:
bronze = spark.table(config.BRONZE)
print(f"Bronze rows: {bronze.count():,}")


## Column reduction + rename

In [ ]:
selected = (
    bronze
    .select(
        col("AIRLINE").alias("airline_name"),
        col("AIRLINE_CODE").alias("airline_code"),
        col("FL_NUMBER").cast("int").alias("fl_number"),
        col("ORIGIN").alias("origin_airport_code"),
        col("DEST").alias("destination_airport_code"),
        to_date(col("FL_DATE")).alias("flight_date"),
        col("CRS_DEP_TIME").cast("int").alias("crs_dep_time"),
        col("CRS_ARR_TIME").cast("int").alias("crs_arr_time"),
        col("CRS_ELAPSED_TIME").cast("double").alias("crs_elapsed_time"),
        col("DISTANCE").cast("double").alias("distance"),
        col("DEP_DELAY").cast("double").alias("dep_delay"),
        col("ARR_DELAY").cast("double").alias("arrival_delay"),
    )
)


## Filters

In [ ]:
enriched = (
    selected
    .withColumn("flight_year", year(col("flight_date")))
    .filter(col("flight_year") != 2020)          # COVID anomaly
    .filter(col("flight_date").isNotNull())
)

print(f"After 2020 filter: {enriched.count():,}")


## Temporal + holiday features — a broadcast join, not four UDFs

The obvious implementation is a Python UDF per flag, and that is what this notebook
did originally. It is also the classic Spark anti-pattern.

**Why it was slow.** A Python UDF cannot run in the JVM. Every row is serialised out
to a Python worker, evaluated, and serialised back — four times per row, across
roughly two million rows. The holiday lookups themselves are cheap; the crossing of
the JVM/Python boundary is not, and it happens eight million times.

**The observation that fixes it.** All four flags are functions of the *date* alone,
and there are only about two thousand distinct dates in the dataset. The same answers
were being recomputed roughly a thousand times each.

**The fix.** Materialise one row per calendar date in the driver — where the Python
cost is paid ~2,000 times instead of ~2,000,000 — and broadcast it. The join runs
entirely in the JVM, and because the dimension is a few hundred kilobytes it ships to
every executor with no shuffle.

`src.features.build_date_dimension` is unit-tested against the very helpers it
replaces (`test_agrees_with_the_udf_helpers_it_replaces` compares every day of 2019),
so this is a provable refactor rather than a rewrite that hopefully matches.

The `dayofweek` convention is the trap: Spark numbers 1=Sunday, Python's
`date.weekday()` numbers 0=Monday. Getting that wrong shifts the feature by a day and
is invisible in aggregates, so `spark_day_of_week` encodes it once and is pinned by
its own test.


In [ ]:
# One row per calendar date spanning the data, built from the unit-tested helpers.
bounds = enriched.select(
    F.min("flight_date").alias("lo"), F.max("flight_date").alias("hi")
).first()
print(f"Date range in Silver: {bounds['lo']} -> {bounds['hi']}")

DATE_DIM_SCHEMA = StructType([
    StructField("flight_date", DateType(), False),
    StructField("flight_month", IntegerType(), False),
    StructField("day_of_week", IntegerType(), False),
    StructField("week_of_year", IntegerType(), False),
    StructField("day_of_month", IntegerType(), False),
    StructField("quarter", IntegerType(), False),
    StructField("is_weekend", IntegerType(), False),
    StructField("is_holiday", IntegerType(), False),
    StructField("is_near_holiday", IntegerType(), False),
    StructField("is_holiday_period", IntegerType(), False),
    StructField("season", StringType(), False),
])

dim_rows = build_date_dimension(bounds["lo"], bounds["hi"])
date_dim = spark.createDataFrame(dim_rows, schema=DATE_DIM_SCHEMA)
print(f"Date dimension: {len(dim_rows):,} rows "
      f"(vs {enriched.count():,} flight rows the UDFs would have touched)")

silver = (
    enriched
    .join(F.broadcast(date_dim), on="flight_date", how="left")
    .withColumn("dep_hour", floor(col("crs_dep_time") / 100))
    .withColumn("arr_hour", floor(col("crs_arr_time") / 100))
)

# A left join silently produces nulls if the dimension fails to cover a date.
# It is built from the observed min/max, so a gap here is a real defect.
uncovered = silver.filter(col("season").isNull()).count()
assert uncovered == 0, f"{uncovered:,} rows joined to no date-dimension row"
print("Date dimension covers every flight row.")


### Does it actually help? Measure, don't assert.

"Broadcast joins are faster than UDFs" is a claim, and a README that makes it without
a number is asking to be doubted. Both implementations run below on the same sample
and the ratio is printed.

The sample keeps this honest *and* cheap: the point is the ratio, and paying for a
full second pass over two million rows to produce a number the sample already gives
would be exactly the kind of unbudgeted compute this project is supposed to avoid.


In [ ]:
BENCH_ROWS = 200_000
bench = enriched.limit(BENCH_ROWS).cache()
bench.count()   # materialise before timing so the cache load is not measured

# --- the original: four Python UDFs, one call per row per flag ----------------
season_udf = udf(get_season, StringType())
holiday_udf = udf(check_holiday, IntegerType())
near_holiday_udf = udf(check_near_holiday, IntegerType())
holiday_period_udf = udf(check_holiday_period, IntegerType())

t0 = time.perf_counter()
(
    bench
    .withColumn("flight_month", spark_month(col("flight_date")))
    .withColumn("day_of_week", dayofweek(col("flight_date")))
    .withColumn("week_of_year", weekofyear(col("flight_date")))
    .withColumn("day_of_month", dayofmonth(col("flight_date")))
    .withColumn("quarter", spark_quarter(col("flight_date")))
    .withColumn("is_weekend", when(col("day_of_week").isin(1, 7), 1).otherwise(0))
    .withColumn("is_holiday", holiday_udf(col("flight_date")))
    .withColumn("is_near_holiday", near_holiday_udf(col("flight_date")))
    .withColumn("is_holiday_period", holiday_period_udf(col("flight_date")))
    .withColumn("season", season_udf(col("flight_month")))
    .write.format("noop").mode("overwrite").save()
)
udf_seconds = time.perf_counter() - t0

# --- the replacement: one broadcast join --------------------------------------
t0 = time.perf_counter()
(
    bench
    .join(F.broadcast(date_dim), on="flight_date", how="left")
    .write.format("noop").mode("overwrite").save()
)
join_seconds = time.perf_counter() - t0

bench.unpersist()

print(f"Sample: {BENCH_ROWS:,} rows")
print(f"  4 Python UDFs   : {udf_seconds:7.2f}s")
print(f"  broadcast join  : {join_seconds:7.2f}s")
print(f"  speedup         : {udf_seconds / max(join_seconds, 1e-9):7.2f}x")
print()
print("`.write.format('noop')` forces full execution without writing anything, so")
print("the timing measures the transformation rather than Spark's laziness or the")
print("cost of a sink.")


## Write, constrain, compact

Three things happen here that the branch previously did none of.

**Table properties.** `optimizeWrite` and `autoCompact` make Delta produce
reasonably-sized files on write instead of leaving a pile of small ones for a later
`OPTIMIZE` to clean up.

**`CHECK` constraints.** The filters above are invariants — 2020 is excluded, dates
are non-null, quarters are 1–4. Expressing them as constraints moves enforcement from
"this notebook did it correctly" to "the table will reject a write that doesn't".
That is the difference between a script and a managed table, and it is what makes a
downstream consumer able to trust the schema. Note `arrival_delay` is deliberately
*not* constrained non-null: cancelled and diverted flights legitimately have none,
and Gold filters them.

**`OPTIMIZE ... ZORDER BY`.** Every downstream reader filters on `flight_year` —
`04_gold` labels off it, `05_train` splits the CV, threshold, and test windows on it.
Z-ordering co-locates those rows so those filters skip files instead of scanning.
Liquid clustering (`CLUSTER BY`) is the newer answer and would be preferable on a
paid workspace; ZORDER is used here because it is dependable on Free Edition.


In [ ]:
(
    silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(config.SILVER)
)

silver_count = spark.table(config.SILVER).count()
print(f"Silver rows: {silver_count:,}")
print(f"Silver columns: {len(spark.table(config.SILVER).columns)}")


In [ ]:
spark.sql(f"""
    ALTER TABLE {config.SILVER} SET TBLPROPERTIES (
        delta.autoOptimize.optimizeWrite = true,
        delta.autoOptimize.autoCompact  = true
    )
""")

# Idempotent: drop before add so re-running the notebook does not fail.
CONSTRAINTS = {
    "flight_date_present": "flight_date IS NOT NULL",
    "covid_year_excluded": "flight_year <> 2020",
    "quarter_in_range": "quarter BETWEEN 1 AND 4",
    "day_of_week_in_range": "day_of_week BETWEEN 1 AND 7",
    "delay_threshold_sane": "crs_elapsed_time IS NULL OR crs_elapsed_time > 0",
}
for name, expr in CONSTRAINTS.items():
    spark.sql(f"ALTER TABLE {config.SILVER} DROP CONSTRAINT IF EXISTS {name}")
    spark.sql(f"ALTER TABLE {config.SILVER} ADD CONSTRAINT {name} CHECK ({expr})")
    print(f"  constraint {name:<22} {expr}")

print("\nEvery constraint validated against the data already in the table —")
print("ALTER TABLE ADD CONSTRAINT fails if any existing row violates it, so the")
print("fact that this cell completed is itself the data-quality assertion.")


In [ ]:
spark.sql(f"OPTIMIZE {config.SILVER} ZORDER BY (flight_year, origin_airport_code)")
display(spark.sql(f"DESCRIBE HISTORY {config.SILVER}").select(
    "version", "timestamp", "operation", "operationMetrics"
).limit(10))


The history above is the audit trail: every write, constraint change, and compaction
is a numbered version. It is also what makes the pipeline debuggable after the fact —
`SELECT * FROM silver_flights VERSION AS OF n` reads the table as it stood before a
change, which is how you answer "did this number move because the model changed or
because the data did?"


In [ ]:
history = spark.sql(f"DESCRIBE HISTORY {config.SILVER}")
current = history.agg(F.max("version")).first()[0]
print(f"Current Silver version: {current}")
print(f"Time travel is available: SELECT * FROM {config.SILVER} VERSION AS OF {current}")

quality = spark.table(config.SILVER).agg(
    F.count("*").alias("rows"),
    F.countDistinct("flight_date").alias("distinct_dates"),
    F.sum(F.when(col("arrival_delay").isNull(), 1).otherwise(0)).alias("null_arrival_delay"),
    F.min("flight_date").alias("first_date"),
    F.max("flight_date").alias("last_date"),
).first()

print(f"\n  rows                 {quality['rows']:,}")
print(f"  distinct dates       {quality['distinct_dates']:,}")
print(f"  null arrival_delay   {quality['null_arrival_delay']:,} "
      f"({quality['null_arrival_delay'] / quality['rows']:.2%})  <- cancelled/diverted")
print(f"  date range           {quality['first_date']} -> {quality['last_date']}")
